In [2]:
import pandas as pd
df = pd.read_csv('data/test_invoices.csv')
print("Dataset Shape:", df.shape)
print("-" * 30)
df.info()

Dataset Shape: (2000000, 31)
------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 31 columns):
 #   Column            Dtype  
---  ------            -----  
 0   invoice_id        int64  
 1   invoice_number    str    
 2   invoice_date      str    
 3   due_date          str    
 4   customer_id       str    
 5   customer_name     str    
 6   customer_email    str    
 7   customer_city     str    
 8   customer_state    str    
 9   customer_country  str    
 10  vendor_id         str    
 11  vendor_name       str    
 12  product_id        str    
 13  product_category  str    
 14  product_name      str    
 15  quantity          int64  
 16  unit_price        float64
 17  discount_percent  float64
 18  tax_percent       float64
 19  subtotal          float64
 20  discount_amount   float64
 21  tax_amount        float64
 22  shipping_cost     float64
 23  total_amount      float64
 24  currency          str    
 

In [3]:
print("--- ALL COLUMNS IN DATASET ---")
print(df.columns.tolist())
print("-" * 50)

rename_mapping = {
    'invoice_id': 'Invoice ID',
    'customer_name': 'Customer',
    'invoice_date': 'Invoice Date',
    'due_date': 'Due Date',
    'customer_country': 'Country'
}

df = df.rename(columns=rename_mapping)

print("\n--- RENAMED COLUMNS ---")
print(df[['Invoice ID', 'Customer', 'Invoice Date', 'Due Date', 'Country']].head())

--- ALL COLUMNS IN DATASET ---
['invoice_id', 'invoice_number', 'invoice_date', 'due_date', 'customer_id', 'customer_name', 'customer_email', 'customer_city', 'customer_state', 'customer_country', 'vendor_id', 'vendor_name', 'product_id', 'product_category', 'product_name', 'quantity', 'unit_price', 'discount_percent', 'tax_percent', 'subtotal', 'discount_amount', 'tax_amount', 'shipping_cost', 'total_amount', 'currency', 'payment_method', 'payment_status', 'payment_date', 'order_channel', 'sales_rep', 'is_overdue']
--------------------------------------------------

--- RENAMED COLUMNS ---
   Invoice ID      Customer Invoice Date    Due Date    Country
0     2500001    James Chen   2025-12-01  2025-12-16  Singapore
1     2500002   Diya Sharma   2025-01-04  2025-02-03         UK
2     2500003  Reyansh Nair   2025-07-07  2025-08-06         UK
3     2500004   Arjun Singh   2025-02-17  2025-03-19        USA
4     2500005   Thomas Wang   2024-05-09  2024-05-24        USA


In [4]:

for col in df.columns:
    print(col)


Invoice ID
invoice_number
Invoice Date
Due Date
customer_id
Customer
customer_email
customer_city
customer_state
Country
vendor_id
vendor_name
product_id
product_category
product_name
quantity
unit_price
discount_percent
tax_percent
subtotal
discount_amount
tax_amount
shipping_cost
total_amount
currency
payment_method
payment_status
payment_date
order_channel
sales_rep
is_overdue


In [5]:
final_mapping = {
    'total_amount': 'Amount',
    'currency': 'Currency',
    'payment_date': 'Payment Date/Status'
}
df = df.rename(columns=final_mapping)

print("--- FINAL MAPPED COLUMNS ---")
main_columns = ['Invoice ID', 'Customer', 'Invoice Date', 'Due Date', 'Payment Date/Status', 'Amount', 'Currency', 'Country']
print(df[main_columns].head())
print("-" * 50)
print("\n--- MISSING VALUES CHECK ---")
print(df[main_columns].isnull().sum())
print("\n--- DUPLICATES CHECK ---")
duplicate_count = df.duplicated(subset=['Invoice ID']).sum()
print(f"Total Duplicate Invoices: {duplicate_count}")

--- FINAL MAPPED COLUMNS ---
   Invoice ID      Customer Invoice Date    Due Date Payment Date/Status  \
0     2500001    James Chen   2025-12-01  2025-12-16          2025-12-05   
1     2500002   Diya Sharma   2025-01-04  2025-02-03          2025-01-25   
2     2500003  Reyansh Nair   2025-07-07  2025-08-06          2025-07-21   
3     2500004   Arjun Singh   2025-02-17  2025-03-19          2025-03-11   
4     2500005   Thomas Wang   2024-05-09  2024-05-24          2024-05-24   

      Amount Currency    Country  
0   66614.93      USD  Singapore  
1   56392.01      INR         UK  
2   20879.49      GBP         UK  
3  102823.20      EUR        USA  
4   49557.54      INR        USA  
--------------------------------------------------

--- MISSING VALUES CHECK ---
Invoice ID             0
Customer               0
Invoice Date           0
Due Date               0
Payment Date/Status    0
Amount                 0
Currency               0
Country                0
dtype: int64

--- DUPLI

In [6]:
df['Invoice Date'] = pd.to_datetime(df['Invoice Date'])
df['Due Date'] = pd.to_datetime(df['Due Date'])
df['Payment Date/Status'] = pd.to_datetime(df['Payment Date/Status'])
negative_amounts = df[df['Amount'] < 0].shape[0]
invalid_dates = df[df['Due Date'] < df['Invoice Date']].shape[0]

print("--- AUDIT RESULTS ---")
print(f"Invoices with Negative Amounts: {negative_amounts}")
print(f"Invoices where Due Date is before Invoice Date: {invalid_dates}")
print("-" * 30)

if negative_amounts > 0 or invalid_dates > 0:
    df = df[(df['Amount'] >= 0) & (df['Due Date'] >= df['Invoice Date'])]
    print("Invalid rows have been dropped from the dataset.")

df['days_to_pay'] = (df['Payment Date/Status'] - df['Invoice Date']).dt.days


df['delay'] = (df['Payment Date/Status'] - df['Due Date']).dt.days

df['paid_late_flag'] = df['delay'].apply(lambda x: 1 if x > 0 else 0)
df['open_paid_flag'] = 'Paid'

print("\n--- NEW CALCULATED COLUMNS ---")
print(df[['Invoice ID', 'days_to_pay', 'delay', 'paid_late_flag']].head())

--- AUDIT RESULTS ---
Invoices with Negative Amounts: 0
Invoices where Due Date is before Invoice Date: 0
------------------------------

--- NEW CALCULATED COLUMNS ---
   Invoice ID  days_to_pay  delay  paid_late_flag
0     2500001            4    -11               0
1     2500002           21     -9               0
2     2500003           14    -16               0
3     2500004           22     -8               0
4     2500005           15      0               0


In [7]:
import os

os.makedirs('data/processed', exist_ok=True)

df.to_csv('data/processed/cleaned_invoices.csv', index=False)

print("Data successfully saved to 'data/processed/cleaned_invoices.csv' 🎉")

Data successfully saved to 'data/processed/cleaned_invoices.csv' 🎉
